# STGCN TPU Batched Training for Google Colab

This notebook introduces **Batched Training DataLoaders** to massively accelerate SpatioTemporal GCN training over a Google Colab TPU. Using `torch_xla`, training takes seconds instead of hours.

In [ ]:
# Setup PyTorch XLA for Google Colab TPU
import os
if "COLAB_TPU_ADDR" in os.environ:
    !pip install torch~=2.3.0 torch_xla[tpu]~=2.3.0 -f https://storage.googleapis.com/libtpu-releases/index.html


In [ ]:
# Download the project from GitHub
!git clone https://github.com/saltypal/Hierarchical-multi-agent-RL-for-Urban-Traffic

import sys
import os
# Update PROJECT_ROOT to the cloned directory
PROJECT_ROOT = '/content/Hierarchical-multi-agent-RL-for-Urban-Traffic'
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# PyTorch XLA specific imports
import torch_xla
import torch_xla.core.xla_model as xm

# Import topology and model (Make sure they are accessible in PROJECT_ROOT)
from src.topology import Topology
from src.controllers.area_controller import SpatioTemporalGCN, STGCN_SEQ_LEN


In [ ]:
print("Loading topology...")
topology = Topology(PROJECT_ROOT)
area_id = "HSR_Layout"
n_wards = len(topology.get_area_wards(area_id))

print("Loading dataset...")
# Load the pre-compiled temporal dataset
data_path = f"{PROJECT_ROOT}/models/gnn/global_temporal_data.pt"
dataset = torch.load(data_path, map_location="cpu", weights_only=False)
print(f"Loaded {len(dataset)} raw transitions.")

# Extract relevant samples for the target area
area_samples = [d for d in dataset if d["area_id"] == area_id]
print(f"Extracted {len(area_samples)} samples for {area_id}.")

# Build sequences
sequences = []
targets = []
seq_len = STGCN_SEQ_LEN

for i in range(len(area_samples) - seq_len):
    window = area_samples[i : i + seq_len]
    next_step = area_samples[i + seq_len]
    
    # In area_controller.py, the targets are the congestion at the next step.
    # From AreaForecaster logic: `target_y = y[n, 0]` where 0 is congestion.
    target_y = next_step["features"][:, 0]  # congestion metric index 0 for all wards
    
    x_seq = np.stack([w["features"] for w in window], axis=0)
    sequences.append(x_seq)
    targets.append(target_y)

# Convert to batched tensors
X = torch.tensor(np.array(sequences), dtype=torch.float32)  # [B, T, N, F]
Y = torch.tensor(np.array(targets), dtype=torch.float32)    # [B, N]

print(f"Final sequence tensors: X={X.shape}, Y={Y.shape}")


In [ ]:
# TPU Device Setup
device = xm.xla_device()
print("Using device:", device)

# Define Model & Adjacency Matrix
a_hat = torch.tensor(topology.get_adjacency_matrix(area_id), dtype=torch.float32).to(device)
model = SpatioTemporalGCN(in_features=8, hidden_dim=32, seq_len=STGCN_SEQ_LEN).to(device)

# DataLoader (Batching drastically speeds up training!)
batch_size = 256
train_dataset = TensorDataset(X, Y)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

epochs = 100
losses = []

print("Starting TPU Batched Training...")
start_time = time.time()
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (STGCN supports batched input [B, T, N, F])
        pred = model(batch_x, a_hat)
        
        loss = loss_fn(pred, batch_y)
        loss.backward()
        
        # XLA requires xm.optimizer_step() to sync computational graphs
        xm.optimizer_step(optimizer, barrier=True)
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | MSE Loss: {avg_loss:.6f}")

print(f"Training Complete! Time taken: {time.time() - start_time:.2f}s")


In [ ]:
# Plot the training curve
plt.figure(figsize=(8, 4))
plt.plot(losses, label="STGCN TPU Batched Training Loss", color="#f43f5e", linewidth=2)
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.title(f"TPU Training Curve: {area_id}")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# Save the model
model.eval()
save_path = f"{PROJECT_ROOT}/models/gnn/area_model_stgcn.pt"
# Moving to CPU before saving is a best practice for TPU
xm.save(model.cpu().state_dict(), save_path)
print(f"Model saved to: {save_path}")


In [ ]:
# Download the trained weights to your local machine!
from google.colab import files
files.download(save_path)
print("Downloading trained STGCN weights...")
